In [8]:
import re


def remove_stuttering_annotations(text):
    """
    Remove stuttering annotations from transcribed text to get semantic transcription.

    This produces the output that a speech recognition model should generate
    (i.e., what the speaker intended to say without stuttering events).

    Based on the AS-70 paper preprocessing steps (Section 3.2):
    - Removes stuttering event labels (/b, /p, /r, /i)
    - Removes repeated words/characters in square brackets
    - Removes interjection characters marked with /i
    - Removes punctuation
    """

    # Remove text inside angle brackets
    # Pattern: <text> -> remove the entire angle bracket and its content
    text = re.sub(r"<[^>]+>", "", text)

    # First, identify and remove interjection characters marked with /i
    # Pattern: character/i -> remove the character
    text = re.sub(r"([呃啊嗯哦额])\/i", "", text)

    # Handle square brackets with repetitions
    # Pattern: word[word...] means the word in brackets is repeated (should be removed)
    def remove_repetitions(match):
        before_bracket = match.group(1)
        return before_bracket

    text = re.sub(r"([^[\]]+)\[[^\]]+\]", remove_repetitions, text)

    # Remove remaining stuttering markers: /b, /p, /r, /i (and their combinations)
    text = re.sub(r"/[bpri]+", "", text)

    # Remove punctuation
    text = re.sub(r'[，。、！？；：""' "（）《》【】…—]", "", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", "", text)

    return text.strip()


In [9]:
import soundfile as sf


def slice_wav(src_path, dst_path, start_sec, end_sec, blocksize=1024):
    with sf.SoundFile(src_path) as infile:
        # Infer parameters
        samplerate = infile.samplerate
        channels = infile.channels
        subtype = infile.subtype

        start_frame = int(start_sec * samplerate)
        end_frame = int(end_sec * samplerate)

        infile.seek(start_frame)

        with sf.SoundFile(
            dst_path, "w", samplerate=samplerate, channels=channels, subtype=subtype
        ) as outfile:
            frames_left = end_frame - start_frame
            while frames_left > 0:
                read_frames = min(blocksize, frames_left)
                data = infile.read(read_frames)
                if len(data) == 0:
                    break
                outfile.write(data)
                frames_left -= len(data)


In [10]:
from pathlib import Path

from rich.console import Console
from rich.progress import (
    BarColumn,
    Progress,
    SpinnerColumn,
    TextColumn,
    TimeRemainingColumn,
)

# Create console configured for Jupyter
console = Console(force_jupyter=True)

raw_audio_dir = Path("/home/benji/dev/ssa/data/as70/raw/Audio")
processed_audio_dir = Path("/home/benji/dev/ssa/data/as70/audio")
if not processed_audio_dir.exists():
    processed_audio_dir.mkdir(parents=True)

# Collect all files first
annotation_files = []
for path in sorted(Path("/home/benji/dev/ssa/data/as70/raw/Annotation").rglob("*.txt")):
    if not path.is_file():
        continue
    if "_" in path.stem and path.stem.split("_")[1] == "B":
        continue  # skip over stuttering event annotations
    annotation_files.append(path)

processed_rows = []
with Progress(
    SpinnerColumn(),
    TextColumn("[progress.description]{task.description}"),
    BarColumn(),
    TextColumn("[progress.percentage]{task.percentage:>3.0f}%"),
    TimeRemainingColumn(),
    console=console,
) as progress:
    # Outer progress bar for files
    file_task = progress.add_task(
        "[cyan]Processing annotation files...", total=len(annotation_files)
    )

    for path in annotation_files:
        subject_id = path.parent.stem
        lines = path.read_text(encoding="utf-8").splitlines()

        for i, line in enumerate(lines):
            parts = line.split("\t")
            assert len(parts) == 3

            file_type = (
                "C"  # command
                if path.stem[0] == "P"
                else "A"  # speaker A
                if path.stem[-1] == "A"
                else "B"  # speaker B
            )

            clip_id = f"{subject_id}_{file_type}_{i:03d}"
            wav_path = raw_audio_dir / f"{subject_id}.wav"
            slice_wav_path = processed_audio_dir / f"{clip_id}.wav"
            start_sec = float(parts[0])
            end_sec = float(parts[1])
            text = parts[2]

            # slice_wav(wav_path, slice_wav_path, start_sec, end_sec)

            unannotated_text = remove_stuttering_annotations(text)

            relative_slice_wav_path = slice_wav_path.relative_to(Path("/home/benji/dev/ssa"))

            row = {
                "annotated_text": text,
                "unannotated_text": unannotated_text,
                "audio_path": str(relative_slice_wav_path),
                "subject_id": subject_id,
                "clip_id": clip_id,
                "file_type": "Speaker A"
                if file_type == "A"
                else "Speaker B"
                if file_type == "B"
                else "Command",
                "start_sec": start_sec,
                "end_sec": end_sec,
            }
            processed_rows.append(row)

        # Update outer progress bar
        progress.update(file_task, advance=1)


Output()

In [11]:
import polars as pl

df = pl.DataFrame(processed_rows)
df.head()

annotated_text,unannotated_text,audio_path,subject_id,clip_id,file_type,start_sec,end_sec
str,str,str,str,str,str,f64,f64
"""大/r[大]家好，我是叫<姓名>，那。""","""大家好我是叫那""","""data/as70/audio/0001_A_000.wav""","""0001""","""0001_A_000""","""Speaker A""",87.97,94.85
"""我是，今年是，<年龄>岁来自<居住地>，呃/i。""","""我是今年是岁来自""","""data/as70/audio/0001_A_001.wav""","""0001""","""0001_A_001""","""Speaker A""",95.58,102.83
"""我是。""","""我是""","""data/as70/audio/0001_A_002.wav""","""0001""","""0001_A_002""","""Speaker A""",104.09,105.68
"""资深/p的口/b[口/r/b]吃患者。""","""资深的口吃患者""","""data/as70/audio/0001_A_003.wav""","""0001""","""0001_A_003""","""Speaker A""",106.3,119.12
"""我是从小就有口吃。""","""我是从小就有口吃""","""data/as70/audio/0001_A_004.wav""","""0001""","""0001_A_004""","""Speaker A""",119.13,124.49


In [12]:
df.write_parquet("/home/benji/dev/ssa/data/as70/as70.parquet")